# Component 1: remote inference for the requirement drafts (Colab)

This notebook serves the fine-tuned `c1-req` model from a Google Colab GPU, so the C1 app on your laptop can write drafts there instead of on the laptop. It does not change the model.

## Read this first

- **Colab terms.** Google may restrict serving a web service through a tunnel from a Colab runtime. Check the current Colab terms of service before you use this, and use it only if they allow it. This notebook is a convenience, not a permission.
- **Your requirement text leaves your laptop.** Every requirement the app sends for a draft travels through a public tunnel to a Colab machine. Do not use it for documents you are not allowed to send to a third party.
- **The tunnel URL has no password and must not be shared.** Anyone who has the `https://....trycloudflare.com` address can use your model and send it text. Do not paste it into chats, tickets, screenshots or commits.
- **Stop the runtime afterwards.** Run the last cell, then use *Runtime > Disconnect and delete runtime*. A forgotten runtime keeps the tunnel open and uses your Colab quota.

## Steps

1. Choose a GPU: *Runtime > Change runtime type > T4 GPU* (or better).
2. Run the cells below in order, one at a time. Cell 6 prints the tunnel URL; cell 7 checks it; cell 8 stops everything.
3. In the C1 app: Process > gear > paste the URL into *Cloud URL*, tick the consent box, press *Test connection*, then *Save* and choose *Cloud*.
4. When you are done: run the last cell and delete the runtime.

The first request through the tunnel can be slow while the model loads into GPU memory. Keep `LLM_TIMEOUT` at 600 on the laptop (scripts/run_c1_live.ps1 already does).

In [ ]:
# Cell 2: Copy the model from Google Drive
#
# Mounts your Drive and copies My Drive/Reasech Dataset/outputs/c1_merged_q4_k_m.gguf to /content/ (local disk is much faster than Drive).
from google.colab import drive
import os
import shutil

drive.mount("/content/drive")

SRC = "/content/drive/My Drive/Reasech Dataset/outputs/c1_merged_q4_k_m.gguf"
DST = "/content/c1_merged_q4_k_m.gguf"

if not os.path.exists(SRC):
    raise FileNotFoundError("Model file not found in Drive: " + SRC)
if not os.path.exists(DST):
    shutil.copy(SRC, DST)
print("Model file:", DST, round(os.path.getsize(DST) / 1e9, 2), "GB")

In [ ]:
%%bash
# Cell 3: Install Ollama
#
# Installs zstd (the Ollama installer needs it) and then runs the official install script from ollama.com.
set -e
command -v zstd >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq zstd)
curl -fsSL https://ollama.com/install.sh | sh
ollama --version

In [ ]:
# Cell 4: Start the Ollama server
#
# Starts ollama serve in the background with 4 parallel requests and the model kept in memory for 24 hours, then waits until it answers.
import os
import subprocess
import time
import urllib.request

env = dict(os.environ, OLLAMA_NUM_PARALLEL="4", OLLAMA_KEEP_ALIVE="24h")
ollama_log = open("/content/ollama.log", "wb")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=ollama_log, stderr=subprocess.STDOUT)

for _ in range(90):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2).read()
        print("Ollama is answering.")
        break
    except Exception:
        time.sleep(1)
else:
    print(open("/content/ollama.log", errors="replace").read()[-3000:])
    raise RuntimeError("Ollama did not start within 90 seconds (log above).")

In [ ]:
%%bash
# Cell 5: Create the c1-req model
#
# The Modelfile is the one in the repository (project/models/component1/part_b/Modelfile), word for word, except that the FROM line points at the copied file. There is no ADAPTER, and no num_gpu or num_ctx tuning.
set -e
cat > /content/Modelfile <<'MODELFILE_END'
FROM /content/c1_merged_q4_k_m.gguf
TEMPLATE """<|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

{{ .System }}<|eot_id|><|start_header_id|>user<|end_header_id|>

{{ .Prompt }}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{{ .Response }}<|eot_id|>"""
PARAMETER stop "<|eot_id|>"
PARAMETER temperature 0
PARAMETER num_predict 600
MODELFILE_END

ollama create c1-req -f /content/Modelfile
ollama list

In [ ]:
# Cell 6: Open the tunnel
#
# Downloads cloudflared (linux-amd64) from the official GitHub release, opens a quick tunnel to http://localhost:11434 in the background and prints the public address. Keep this address private.
import re
import subprocess
import time

CLOUDFLARED = "/content/cloudflared"
if not os.path.exists(CLOUDFLARED):
    subprocess.run(
        ["curl", "-fsSL", "-o", CLOUDFLARED, "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        check=True,
    )
    os.chmod(CLOUDFLARED, 0o755)

tunnel_log_path = "/content/cloudflared.log"
tunnel_log = open(tunnel_log_path, "wb")
# The Host header is set to localhost:11434 because Ollama checks it.
tunnel_proc = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--no-autoupdate", "--http-host-header", "localhost:11434", "--url", "http://localhost:11434"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

TUNNEL_URL = None
for _ in range(90):
    text = open(tunnel_log_path, errors="replace").read()
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if found:
        TUNNEL_URL = found.group(0)
        break
    time.sleep(1)

if TUNNEL_URL is None:
    print(open(tunnel_log_path, errors="replace").read()[-3000:])
    raise RuntimeError("No tunnel address appeared within 90 seconds (log above).")

print("Paste this into the C1 app (Process > gear > Cloud URL). Do not share it:")
print(TUNNEL_URL)

In [ ]:
# Cell 7: Check the tunnel
#
# Calls the tunnel from here with /api/tags and /v1/models (the path the C1 app uses) and prints the models. c1-req must be in both lists. A new tunnel address can take a few seconds to resolve, so this retries for about 60 seconds.
import json
import time
import urllib.request


def get_json(path):
    req = urllib.request.Request(TUNNEL_URL + path, headers={"User-Agent": "c1-check/1.0"})
    with urllib.request.urlopen(req, timeout=20) as response:
        return json.load(response)


last_error = None
for _ in range(12):
    try:
        tags = get_json("/api/tags")
        models = get_json("/v1/models")
        break
    except Exception as exc:
        last_error = exc
        time.sleep(5)
else:
    raise RuntimeError("The tunnel did not answer: " + repr(last_error))

print("/api/tags  :", [m["name"] for m in tags.get("models", [])])
print("/v1/models :", [m["id"] for m in models.get("data", [])])
assert any(m["id"].split(":")[0] == "c1-req" for m in models.get("data", [])), "c1-req is not served"
print("OK: c1-req is served through the tunnel.")

In [ ]:
# Cell 8: Stop everything
#
# Stops the tunnel and Ollama. Afterwards use Runtime > Disconnect and delete runtime so nothing keeps running.
import subprocess

for name in ("tunnel_proc", "ollama_proc"):
    proc = globals().get(name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
subprocess.run(["pkill", "-f", "ollama serve"], check=False)
print("Tunnel and Ollama stopped. Now use Runtime > Disconnect and delete runtime.")